# FloodWatch: manual-assisted wheel depth experiment

Run the code cell below, then upload one JPG or PNG. CPU is sufficient; no API key or model training is required. The sliders let you manually mark a wheel and its local waterline. This first experiment isolates the depth calculation from detector errors; it does not call your Roboflow wheel model.

## How to use
1. Set the red box around the full outside tire. Move the top and bottom sliders carefully. Do not use only the rim or only the unsubmerged portion.
2. Set the cyan waterline where water meets that wheel. The line is a local reference, not a water boundary across the entire scene.
3. Confirm the full tire boundaries are reliable and the photo is approximately side-on with little tilt, with the tire resting on the road. If the hidden bottom cannot be located, leave the confirmation unchecked.
4. Enter a measured outside tire diameter only when known. Unknown diameter produces only a submerged-height percentage, not centimeters. Rim diameter is not outside tire diameter.
5. Re-run the cell to upload another photo. Sliders refer to the displayed, orientation-corrected image, so their coordinates stay consistent.

## Interpretation
Approximate depth = outside tire diameter × (wheel bottom − waterline) / (wheel bottom − wheel top). This estimates vertical submerged height, not submerged area. It assumes a suitable view and approximately uniform image scale over the tire. Camera perspective, tire deformation, inaccurate boundaries, and uncertain size introduce error. No depth is calculated when the waterline lies outside the tire height.

You can try the interface on a dry wheel photo, but moving a line over that photo is only a simulated example, not evidence of an actual flood depth. Evaluating accuracy still requires known depth and size, potentially using a controlled setup instead of a flooded street. This notebook is an unvalidated experiment, not an automatic flood-depth system.

Widget reference: https://ipywidgets.readthedocs.io/en/latest/reference/ipywidgets.html


In [ ]:
from google.colab import files, output
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display, clear_output
import ipywidgets as widgets
import io, math

output.enable_custom_widget_manager()

def estimate(top, bottom, waterline, reliable, suitable, diameter=None):
    if not reliable or not suitable:
        return None, 'Unable to estimate: confirm the full tire boundaries and suitable view first.'
    if bottom <= top:
        return None, 'Wheel bottom must be below wheel top.'
    if not top <= waterline <= bottom:
        return None, 'Waterline is outside the tire height. No depth estimate for this setup.'
    fraction = (bottom - waterline) / (bottom - top)
    if diameter is not None and (not math.isfinite(diameter) or diameter <= 0):
        return None, 'Enter a positive measured outside tire diameter, or turn off Known diameter.'
    depth = None if diameter is None else fraction * diameter
    return {'height_fraction': fraction, 'depth_cm': depth}, ''

print('Choose ONE JPG or PNG. Use a side-on wheel photo with little camera tilt.')
uploaded = files.upload()
if not uploaded:
    raise ValueError('No photo uploaded. Run this cell again.')
filename = next(iter(uploaded))
with Image.open(io.BytesIO(uploaded[filename])) as source:
    photo = ImageOps.exif_transpose(source).convert('RGB')
photo.thumbnail((900, 650))
w, h = photo.size
if min(w, h) < 10:
    raise ValueError('Image is too small.')

def slider(label, value, maximum):
    return widgets.IntSlider(description=label, value=value, min=0, max=maximum,
                             continuous_update=False,
                             style={'description_width': '120px'},
                             layout=widgets.Layout(width='95%'))

left = slider('Wheel left', w // 3, w-1)
right = slider('Wheel right', 2*w // 3, w-1)
top = slider('Wheel top', h // 3, h-1)
bottom = slider('Wheel bottom', 2*h // 3, h-1)
water = slider('Waterline', h // 2, h-1)
reliable = widgets.Checkbox(description='I can locate the FULL tire top and bottom reliably.', indent=False)
suitable = widgets.Checkbox(description='Side-on view, little camera tilt; tire rests on the road.', indent=False)
known = widgets.Checkbox(description='I know the measured outside tire diameter.', indent=False)
diameter = widgets.FloatText(value=0, description='Diameter (cm)', disabled=True,
                            style={'description_width': '120px'})
preview = widgets.Output()

def render(change=None):
    diameter.disabled = not known.value
    canvas = photo.copy()
    draw = ImageDraw.Draw(canvas)
    valid_box = left.value < right.value and top.value < bottom.value
    if valid_box:
        draw.rectangle((left.value, top.value, right.value, bottom.value), outline='red', width=3)
    draw.line((0, water.value, w-1, water.value), fill='cyan', width=3)
    draw.text((8, max(0, water.value-15)), 'Waterline', fill='blue', stroke_width=1, stroke_fill='white')
    result, reason = estimate(top.value, bottom.value, water.value,
                              reliable.value, suitable.value,
                              diameter.value if known.value else None)
    with preview:
        clear_output(wait=True)
        display(canvas)
        print('Manual-assisted estimate — not automatic water detection.')
        if not valid_box:
            print('Move the boundaries so left < right and top < bottom.')
        elif reason:
            print(reason)
        else:
            print(f"Estimated submerged HEIGHT: {100*result['height_fraction']:.1f}%")
            if result['depth_cm'] is None:
                print('Depth in centimeters unavailable: outside tire diameter is unknown.')
            else:
                print(f"Approximate depth at this tire: {result['depth_cm']:.1f} cm")
        print('Red box = full outside tire. Cyan line = water surface at that tire.')

for control in (left, right, top, bottom, water, reliable, suitable, known, diameter):
    control.observe(render, names='value')

display(widgets.HTML('<b>1.</b> Position the red box around the full tire, not just the rim or visible part.<br>'
                     '<b>2.</b> Move the cyan line to the water surface where it meets this tire.<br>'
                     '<b>3.</b> Confirm the boundaries/view only if justified. Leave diameter unknown unless measured.<br>'
                     'If the submerged bottom is hidden and cannot be located reliably, leave the first checkbox off.'))
display(widgets.VBox([left, right, top, bottom, water, reliable, suitable, known, diameter, preview]))
render()
